# Frozen evaluation

Run this only after choosing a checkpoint using a held-out development split. Never use the frozen MATE or puzzle scores to tune hyperparameters. The evaluator reports MATE and the official full-solution-sequence puzzle metric.

In [ ]:
from pathlib import Path
import os, subprocess, sys, glob
REPO = Path('/kaggle/working/chess-slm-benchmark')
if not REPO.exists():
    subprocess.run(['git', 'clone', 'https://github.com/Vedang-P/chess-slm-benchmark.git', str(REPO)], check=True)
else:
    subprocess.run(['git', '-C', str(REPO), 'fetch', 'origin'], check=True)
    subprocess.run(['git', '-C', str(REPO), 'reset', '--hard', 'origin/main'], check=True)
SL_REPO = Path('/kaggle/working/searchless_chess')
if not SL_REPO.exists():
    subprocess.run(['git', 'clone', 'https://github.com/google-deepmind/searchless_chess.git', str(SL_REPO)], check=True)
MATE = REPO / 'data/positions'
PUZZLES = SL_REPO / 'data/puzzles.csv'
assert (REPO / 'scripts/eval_gavn.py').exists()
subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', 'python-chess', 'pandas'], check=True)
RUN_ID = 'account1-gavn-3m-seed0'  # change per eval; pulls LATEST checkpoint from HF
if not os.environ.get('HF_WRITE_TOKEN'):
    for _p in sorted(glob.glob('/kaggle/input/*/hf_token.txt')):
        os.environ['HF_WRITE_TOKEN'] = Path(_p).read_text().strip()
        break
import sys as _sys
_sys.path.insert(0, str(REPO))
from scripts.kaggle_checkpoint import api, download_latest
out = Path('/kaggle/working/eval-ckpt')
CKPT = download_latest(api(REPO), 'vedangfake/chess-slm-benchmark', RUN_ID, out)
assert CKPT is not None and CKPT.exists(), 'no remote checkpoint found'
print('evaluating', CKPT)
os.chdir(REPO)

In [ ]:
mate_files = ','.join(str(p) for p in sorted(MATE.glob('*.json')))
cmd = [sys.executable, 'scripts/eval_gavn.py', '--checkpoint', str(CKPT), '--sl-repo', str(SL_REPO), '--eval', mate_files, '--puzzles', str(PUZZLES), '--num-puzzles', '10000', '--score', 'q']
print(' '.join(cmd))
subprocess.run(cmd, check=True)

For the paper, preserve the complete stdout, checkpoint hash, git commit, dataset hash, GPU type, seed, and exact command.